# 情報基礎 対面3 ── 学習データの外側としきい値

受講者が提出した手書きの数字で，数字を識別するプログラムを学習させる．学習に使っていない数字と，数字でないもの（アルファベットと模様）を入力し，出力される確率を調べる．

**進め方**

- 上のセルから順に，セルの左の ▶（実行ボタン）を押す．コードは書き換えない
- 3 と 5 には入力欄がある（セルの右側）．16進数を貼ったら，そのセルの ▶ を押す
- `NameError` が出たら，1 のセルから順に実行し直す
- ほかのエラーが出たら，まず最後の1行を読む．エラーの種類と内容が表示される

## 1 データの読み込み

次のセルを実行する．受講者が提出した 8×8 の絵を読み込む．

- 学習データ：学習に使う数字
- テストデータ：学習に使わず，正解率を調べるために残した数字（0〜9 の各3個）
- 数字でないもの：アルファベットと模様

In [ ]:
import urllib.request
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/kazumichiINOUE/info-basics-notebooks/7ee53a7c8b8f1dfcaf74a7515bc6f65927dc963e/data/k1_g1.txt"


def to_bits(h):
    """16進数16文字を，64個の 0 と 1 の列にする"""
    v = int(h, 16)
    return np.array([v >> (63 - i) & 1 for i in range(64)], dtype=float)


data = {"train": [], "test": [], "other": []}
for line in urllib.request.urlopen(URL).read().decode("utf-8").splitlines():
    if line and not line.startswith("#"):
        kind, label, h = line.split()
        data[kind].append((label, h))

train_x = np.array([to_bits(h) for _, h in data["train"]])
train_t = np.array([int(a) for a, _ in data["train"]])
test_x = np.array([to_bits(h) for _, h in data["test"]])
test_t = np.array([int(a) for a, _ in data["test"]])
other_x = np.array([to_bits(h) for _, h in data["other"]])
other_label = [a for a, _ in data["other"]]
print("学習データ　　　", len(train_x), "個")
print("テストデータ　　", len(test_x), "個")
print("数字でないもの　", len(other_x), "個")

## 2 学習

次のセルを実行する．入力 64個 → 中間層 16個 → 出力 10個 のプログラム（以下，64→16→10）を，学習データで学習させる．数秒かかる．

- 出力の10個は，0〜9 のそれぞれである確率．10個の和は 1（ソフトマックス関数）
- パラメータの初期値は乱数．乱数の種を固定してあるので，全員が同じ結果になる
- 学習データ全体で 3000エポック学習させる

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def train_mlp(x, t):
    """64→16→10 を学習させ，入力から 10個の確率を計算する関数を返す"""
    rng = np.random.default_rng(0)       # 乱数の種
    W1 = rng.normal(0, 0.3, (64, 16))    # 入力 → 中間層の重み
    b1 = np.zeros(16)                    # 中間層のバイアス
    W2 = rng.normal(0, 0.3, (16, 10))    # 中間層 → 出力の重み
    b2 = np.zeros(10)                    # 出力のバイアス
    lr = 0.5                             # 学習率

    def predict(x):
        h = sigmoid(x @ W1 + b1)
        z = h @ W2 + b2
        e = np.exp(z - z.max(axis=-1, keepdims=True))   # ソフトマックス関数
        return h, e / e.sum(axis=-1, keepdims=True)

    target = np.eye(10)[t]               # 正解ラベルを 1，ほかを 0 にした表
    for epoch in range(3000):
        h, y = predict(x)
        d = (y - target) / len(x)        # 出力と正解の差
        dh = d @ W2.T * h * (1 - h)
        W2 -= lr * h.T @ d
        b2 -= lr * d.sum(axis=0)
        W1 -= lr * x.T @ dh
        b1 -= lr * dh.sum(axis=0)
    return lambda x: predict(x)[1]


prob_mlp = train_mlp(train_x, train_t)
print("学習データでの正解率　", f"{(prob_mlp(train_x).argmax(axis=1) == train_t).mean():.0%}")
print("テストデータでの正解率", f"{(prob_mlp(test_x).argmax(axis=1) == test_t).mean():.0%}")

## 3 1個の絵の確率

次のセルを実行する．絵と，その 10個の確率を表示する関数を定義するだけなので，何も表示されない．

In [ ]:
def bars(ax, y, title):
    ax.bar(range(10), y, color=["#c0392b" if k == y.argmax() else "#8a857e" for k in range(10)])
    ax.set_xticks(range(10))
    ax.set_ylim(0, 1)
    ax.set_title(title)


def show(h):
    """16進数の絵と，64→16→10 の 10個の確率を表示する"""
    x = to_bits(h)
    y = prob_mlp(x[None])[0]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3), gridspec_kw={"width_ratios": [1, 2]})
    ax1.imshow(x.reshape(8, 8), cmap="Greys", vmin=0, vmax=1)
    ax1.set_xticks([])
    ax1.set_yticks([])
    bars(ax2, y, "64-16-10")
    plt.show()
    print("確率が最大のラベル", y.argmax(), "　確率", round(float(y.max()), 3))

[「8×8グリッド」のページ](https://webclass.gks.saitama-u.ac.jp/webclass/text/26/26Y70013/a4fc6b8717fd413334c3b66940f3636f.html)で絵を描き，「コピーする」で16進数をコピーする．次のセルの入力欄 `h` に貼り（初めの16進数は消す），▶ を押す．左に絵，右に 10個の確率が表示される．初めの `3C66666666663C00` は「0」の絵．

In [ ]:
#@title 自分の絵（64→16→10）
h = "3C66666666663C00"  #@param {type:"string"}
show(h.strip())

## 4 データ拡張と CNN

次のセルを実行する．1分ほどかかる．

- データ拡張：学習データの絵に，回転・拡大縮小・平行移動・せん断を乱数で組み合わせた変形を当て，変形した絵を同じラベルで加える．114個 → 3,420個（30倍）
- CNN：畳み込み層2つ（3×3，8組と16組）とプーリングを使うプログラム．データ拡張した学習データで学習させる
- 乱数の種を固定してあるので，全員が同じ結果になる

In [ ]:
import time
import torch
import torch.nn as nn
from PIL import Image


def transform(x, rot, scale, tx, ty, shear, thr=0.4):
    """8×8 の絵を 32×32 に広げて変形し，4×4 ごとの平均が thr 以上のマスを 1 として 8×8 に戻す"""
    big = np.kron(x.reshape(8, 8), np.ones((4, 4))).astype(np.float32)
    a = np.deg2rad(rot)
    c, s = np.cos(a), np.sin(a)
    A = np.array([[c, -s], [s, c]]) @ np.array([[1, shear], [0, 1]]) / scale
    off = np.array([16, 16]) - A @ np.array([16 + tx, 16 + ty])
    im = Image.fromarray(big).transform((32, 32), Image.Transform.AFFINE,
                                        (A[0, 0], A[0, 1], off[0], A[1, 0], A[1, 1], off[1]),
                                        Image.Resampling.BILINEAR)
    return (np.asarray(im).reshape(8, 4, 8, 4).mean(axis=(1, 3)) >= thr).ravel().astype(np.float32)


def augment(X, Y, n, seed=0):
    """元の絵と，変形した絵 n−1 組を重ねる（n 倍）．変形の大きさは絵ごとに乱数で決める"""
    rng = np.random.default_rng(seed)
    out = [X.astype(np.float32)]
    for _ in range(n - 1):
        batch = []
        for x in X:
            rot = rng.uniform(-15, 15); scale = rng.uniform(0.85, 1.1); shear = rng.uniform(-0.2, 0.2)
            tx, ty = rng.uniform(-1, 1, 2) * 4
            batch.append(transform(x, rot, scale, tx, ty, shear))
        out.append(np.array(batch))
    return np.vstack(out), np.tile(Y, n)


start = time.time()
aug_x, aug_t = augment(train_x, train_t, 30)
print("データ拡張した学習データ", len(aug_x), "個")

torch.set_num_threads(4)
torch.manual_seed(0)
cnn = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.Linear(64, 10))
opt = torch.optim.Adam(cnn.parameters(), 3e-3)
xs = torch.tensor(aug_x, dtype=torch.float32).view(-1, 1, 8, 8)
ts = torch.tensor(aug_t)
g = torch.Generator().manual_seed(0)
for step in range(1500):                 # 256個ずつ 1,500回修正する
    idx = torch.randint(0, len(xs), (256,), generator=g)
    opt.zero_grad()
    nn.functional.cross_entropy(cnn(xs[idx]), ts[idx]).backward()
    opt.step()


def prob_cnn(x):
    with torch.no_grad():
        z = cnn(torch.tensor(np.asarray(x), dtype=torch.float32).view(-1, 1, 8, 8))
    return torch.softmax(z, 1).numpy()


print("CNN のパラメータ", sum(p.numel() for p in cnn.parameters()), "個")
print("CNN のテストデータでの正解率", f"{(prob_cnn(test_x).argmax(axis=1) == test_t).mean():.0%}")
print("かかった時間", round(time.time() - start), "秒")

比べるため，次のセルを実行する．64→16→10 を，データ拡張した 3,420個で学習させる．10秒ほどかかる．

In [ ]:
prob_mlp_aug = train_mlp(aug_x, aug_t)
for name, f in [("64→16→10　　　　　　　", prob_mlp), ("64→16→10 ＋ データ拡張", prob_mlp_aug), ("CNN ＋ データ拡張　　　", prob_cnn)]:
    print(name, "テストデータでの正解率", f"{(f(test_x).argmax(axis=1) == test_t).mean():.0%}")

## 5 数字でないもの

次のセルを実行する．前回提出されたアルファベットと模様から 24個を表示する．絵の上は元のラベル（模様は *）．絵の下の2行は，上が 64→16→10，下が CNN の「確率が最大のラベル（その確率）」．0.9 以上は赤．

In [ ]:
y_other_mlp, y_other_cnn = prob_mlp(other_x), prob_cnn(other_x)
pick = np.linspace(0, len(other_x) - 1, 24).astype(int)
fig, axes = plt.subplots(3, 8, figsize=(12, 6.5))
for ax, i in zip(axes.flat, pick):
    ax.imshow(other_x[i].reshape(8, 8), cmap="Greys", vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(other_label[i], fontsize=11)
    for k, y in enumerate([y_other_mlp[i], y_other_cnn[i]]):
        ax.text(0.5, -0.12 - 0.16 * k, f"{y.argmax()} ({y.max():.2f})", transform=ax.transAxes, ha="center", va="top",
                fontsize=10, color="#c0392b" if y.max() >= 0.9 else "#444444")
plt.subplots_adjust(hspace=0.8)
plt.show()
for name, y in [("64→16→10", y_other_mlp), ("CNN", y_other_cnn)]:
    print(name, "：数字でないもの", len(other_x), "個のうち，最大の確率が 0.9 以上のもの", (y.max(axis=1) >= 0.9).sum(), "個")

自分で描いたアルファベットを，2つのプログラムに入力する．「8×8グリッド」のページの16進数を，次のセルの入力欄 `h` に貼り，▶ を押す．

In [ ]:
#@title 自分の絵（2つのプログラム）
h = "3C66666666663C00"  #@param {type:"string"}
x = to_bits(h.strip())
fig, axes = plt.subplots(1, 3, figsize=(11, 3), gridspec_kw={"width_ratios": [1, 2, 2]})
axes[0].imshow(x.reshape(8, 8), cmap="Greys", vmin=0, vmax=1)
axes[0].set_xticks([])
axes[0].set_yticks([])
for ax, (name, f) in zip(axes[1:], [("64-16-10", prob_mlp), ("CNN", prob_cnn)]):
    bars(ax, f(x[None])[0], name)
plt.show()
for name, f in [("64→16→10", prob_mlp), ("CNN　　　", prob_cnn)]:
    y = f(x[None])[0]
    print(name, "確率が最大のラベル", y.argmax(), "　確率", round(float(y.max()), 3))